# 03 Surge mass transfer and tributary hypsometry

Volume change of the surge reservoirs and receiving areas (polygons in `../Data_Paper/vectors/surge_areas/`), background-corrected transfer per surge, and area per 100 m elevation band for each RGI 6.0 tributary (Agassiz RGI60-01.23649, Seward RGI60-01.13696, Marvine RGI60-01.23653).

Writes to `../Data_Paper/tables/surge_transfer/`:
- `surge_transfer.csv`: volume change, background and transfer per surge (km³)
- `hypsometry_bands_common.csv`: area per band at the start and end of each interval, pixels valid in both DEMs
- `hypsometry_bins_5dem_footprint.csv`: area per band for each DEM, pixels valid in all five DEMs (Fig. 3)

In [ ]:
import os
import geoutils as gu
import xdem
import numpy as np
import pandas as pd

out_dir = '../Data_Paper/tables/surge_transfer'
os.makedirs(out_dir, exist_ok=True)

## DEMs and time intervals
All DEMs on the 2019 grid (hypsometry). SRTM reference date 1 September 1999.

In [ ]:
dems = {}

dems[2019] = xdem.DEM('../Data_Paper/DEMs/d2019_icesat2.tif')
dems[2019].set_nodata(np.nan)
dems[2022] = xdem.DEM('../Data_Paper/DEMs/d2022_icesat2.tif').reproject(dems[2019])
dems[2022].set_nodata(np.nan)
dems[2012] = xdem.DEM('../Data_Paper/DEMs/d2012_coregistered.tif').reproject(dems[2019])
dems[2012].set_nodata(np.nan)
dems[1999] = xdem.DEM('../Data_Paper/DEMs/d1999_coregistered.tif').reproject(dems[2019])
dems[1999].set_nodata(np.nan)
dems[2002] = xdem.DEM('../Data_Paper/DEMs/d2002_coregistered.tif').reproject(dems[2019])
dems[2002].set_nodata(np.nan)

# For each entry, set values <=0 to 0
for year in dems:
    dems[year].data[dems[year].data <= 0] = np.nan


list_years = sorted(dems.keys())

time_intervals = []
# Calculate the time intervals (SRTM: 1 September 1999)
time_intervals.append((pd.to_datetime('2002-08-04') - pd.to_datetime('1999-09-01')).days / 365.25)
time_intervals.append((pd.to_datetime('2012-08-30') - pd.to_datetime('2002-08-04')).days / 365.25)
time_intervals.append((pd.to_datetime('2019-10-10') - pd.to_datetime('2012-08-30')).days / 365.25)
time_intervals.append((pd.to_datetime('2022-09-18') - pd.to_datetime('2019-10-10')).days / 365.25)
time_intervals.append((pd.to_datetime('2022-09-18') - pd.to_datetime('1999-09-01')).days / 365.25)


## dh/dt, uncertainties and variograms (from step 02)
`calculate_area_mean_and_error`: area-mean dh/dt and its uncertainty, with the number of effective samples from the variogram model.

In [ ]:
# dh/dt, σ(dh/dt) and variogram models from 02_elevation_change.ipynb (same period order as there)
periods = ['2002-1999', '2012-2002', '2019-2012', '2022-2019', '2022-1999']
dhdt, dhdt_sig, dhdt_variograms = {}, {}, {}
for period in periods:
    end, start = period.split('-')
    dhdt[period] = gu.Raster(f'../Data_Paper/elevation_change/dhdt_{end}_{start}.tif')
    dhdt_sig[period] = gu.Raster(f'../Data_Paper/elevation_change/dhdt_sig_{end}_{start}.tif')
    dhdt_variograms[period] = pd.read_csv(f'../Data_Paper/elevation_change/{end}_{start}_variogram.csv', index_col=0)


def calculate_area_mean_and_error(dh, sig_dh, area_mask, params_variogram_model, rasterize_resolution = 1000):

    area_vector = gu.Vector(area_mask.polygonize())

    # Calculate the area-averaged uncertainty with these models
    neff = xdem.spatialstats.number_effective_samples(area=area_vector, params_variogram_model=params_variogram_model, rasterize_resolution = rasterize_resolution)

    # Calculate the mean random error in the area
    mean_sig = np.nanmean(sig_dh[area_mask])

    # Compute the standard error
    sig_mean_dh = mean_sig / np.sqrt(neff)

    # Mean elevation difference
    mean_dh = np.nanmean(dh[area_mask])

    return mean_dh, sig_mean_dh


## Tributary masks and volume change of the surge reservoirs and receiving areas
Mean dh × polygon area (km³) and its uncertainty, for every interval with a `surge_areas/<end>_<start>/` folder.

In [ ]:
### --- Create masks for each tributary --- ###
inlier_mask = gu.Vector('../External_Data/RGI60/01_rgi60_Alaska.shp').reproject(dems[2019]).ds
# Select glaciers of interest using the actual GeoDataFrame (glacier_mask): Seward, Marvine, Agassiz
glaciers_of_interest = inlier_mask[inlier_mask['RGIId'].isin(['RGI60-01.13696', 'RGI60-01.23653', 'RGI60-01.23649'])]
seward_mask = gu.Vector(glaciers_of_interest.loc[glaciers_of_interest['RGIId'] == 'RGI60-01.13696']).create_mask(dems[2019])
agassiz_mask = gu.Vector(glaciers_of_interest.loc[glaciers_of_interest['RGIId'] == 'RGI60-01.23649']).create_mask(dems[2019])
marvine_mask = gu.Vector(glaciers_of_interest.loc[glaciers_of_interest['RGIId'] == 'RGI60-01.23653']).create_mask(dems[2019])

masks = [agassiz_mask, seward_mask, marvine_mask]
mask_names = ['Agassiz', 'Seward', 'Marvine']


dict_surge = {}

for yr, (year_range, data) in enumerate(dhdt.items()):
    dict_surge[year_range] = {}
    year_file = year_range.replace('-', '_')
    if os.path.exists(f"../Data_Paper/vectors/surge_areas/{year_file}/"):
        shapefiles = [f for f in os.listdir(f"../Data_Paper/vectors/surge_areas/{year_file}/") if f.endswith('.shp')]

        glaciers = [file.split('_')[0] for file in shapefiles]
        # Get rid of duplicates
        glaciers = list(set(glaciers))

        for g in glaciers:
            masks_glacier = [shapefile for shapefile in shapefiles if shapefile.startswith(g)]
            mask_reservoir = [m for m in masks_glacier if 'Reservoir' in m]
            mask_receiver = [m for m in masks_glacier if 'Receiver' in m]

            dict_surge[year_range][g] = {}
            dict_surge[year_range][g]['reservoir'] = {}
            dict_surge[year_range][g]['receiver'] = {}

            res_vect = gu.Vector(f"../Data_Paper/vectors/surge_areas/{year_file}/{mask_reservoir[0]}")
            res_area = res_vect.area.values
            rec_vect = gu.Vector(f"../Data_Paper/vectors/surge_areas/{year_file}/{mask_receiver[0]}")
            rec_area = rec_vect.area.values

            dict_surge[year_range][g]['reservoir']['avg'],  dict_surge[year_range][g]['reservoir']['sig'] = calculate_area_mean_and_error(dhdt[year_range]*time_intervals[yr],
                                                                                                                dhdt_sig[year_range]*time_intervals[yr],
                                                                                                                res_vect.create_mask(dhdt[year_range]),
                                                                                                                dhdt_variograms[year_range], rasterize_resolution = 500)
            dict_surge[year_range][g]['reservoir']['avg'] = (dict_surge[year_range][g]['reservoir']['avg'] * res_area) / 1e9
            dict_surge[year_range][g]['reservoir']['sig'] = (dict_surge[year_range][g]['reservoir']['sig'] * res_area) / 1e9
            dict_surge[year_range][g]['reservoir']['area'] = res_area[0] / 1e6  # Convert to km²
            


            dict_surge[year_range][g]['receiver']['avg'], dict_surge[year_range][g]['receiver']['sig'] = calculate_area_mean_and_error(dhdt[year_range]*time_intervals[yr],
                                                                                                                dhdt_sig[year_range]*time_intervals[yr],
                                                                                                                rec_vect.create_mask(dhdt[year_range]),
                                                                                                                dhdt_variograms[year_range], rasterize_resolution = 500)
            dict_surge[year_range][g]['receiver']['avg'] = (dict_surge[year_range][g]['receiver']['avg'] * rec_area) / 1e9
            dict_surge[year_range][g]['receiver']['sig'] = (dict_surge[year_range][g]['receiver']['sig'] * rec_area) / 1e9
            dict_surge[year_range][g]['receiver']['area'] = rec_area[0] / 1e6  # Convert to km²

## Background-corrected surge transfer → `surge_transfer.csv`
T = volume transferred reservoir → receiving area. B = mean background rate over the polygon (background period) × polygon area × dt of the surge interval. T_res = −dV_res + B_res; T_sink = dV_sink − B_sink.

In [ ]:
_per_idx = {k: i for i, k in enumerate(dhdt)}  # dhdt keys in time_intervals order
surges = [('2022-2019', 'Seward', '2019-2012'), ('2002-1999', 'Agassiz', '2019-2012'),
          ('2002-1999', 'Marvine', '2012-2002'), ('2019-2012', 'Marvine', '2012-2002')]
rows_T = []
for per, g, bg in surges:
    dt = time_intervals[_per_idx[per]]
    row = {'surge': f'{g} {per}', 'background_period': bg, 'dt_surge_a': dt}
    for part, fname in [('res', 'Reservoir'), ('sink', 'Receiver')]:
        vect = gu.Vector(f"../Data_Paper/vectors/surge_areas/{per.replace('-', '_')}/{g}_{fname}.shp")
        area = float(vect.area.values[0])
        key = 'reservoir' if part == 'res' else 'receiver'
        dV, sdV = float(dict_surge[per][g][key]['avg'][0]), float(dict_surge[per][g][key]['sig'][0])
        r, sr = calculate_area_mean_and_error(dhdt[bg], dhdt_sig[bg], vect.create_mask(dhdt[bg]), dhdt_variograms[bg], rasterize_resolution=500)
        r, sr = float(r), float(np.ravel(sr)[0])
        B, sB = r * area * dt / 1e9, sr * area * dt / 1e9
        row.update({f'area_{part}_km2': area / 1e6, f'bg_rate_{part}': r, f'bg_rate_sig_{part}': sr,
                    f'dV_{part}': dV, f'sig_dV_{part}': sdV, f'B_{part}': B, f'sig_B_{part}': sB})
    row['T_res'] = -row['dV_res'] + row['B_res']
    row['sig_T_res'] = np.hypot(row['sig_dV_res'], row['sig_B_res'])
    row['T_sink'] = row['dV_sink'] - row['B_sink']
    row['sig_T_sink'] = np.hypot(row['sig_dV_sink'], row['sig_B_sink'])
    d, sd = row['T_res'] - row['T_sink'], np.hypot(row['sig_T_res'], row['sig_T_sink'])
    row['diff_res_minus_sink'] = d; row['sig_diff'] = sd; row['agree_1sigma'] = abs(d) <= sd; row['agree_2sigma'] = abs(d) <= 2 * sd
    rows_T.append(row)
surge_T = pd.DataFrame(rows_T)
surge_T.to_csv(f'{out_dir}/surge_transfer.csv', index=False)
surge_T.T

## Area per 100 m band, pixels valid in both DEMs of each comparison → `hypsometry_bands_common.csv`
1999-2022, then each consecutive interval; 0-1000 m a.s.l., whole RGI tributary mask.

In [ ]:
bins = np.arange(0, 1100, 100)
band_areas = {}
for y0, y1 in [(list_years[0], list_years[-1])] + list(zip(list_years[:-1], list_years[1:])):
    for mask, name in zip(masks, mask_names):
        # Apply mask to both matrices
        dem1 = dems[y0].data[mask.data]
        dem2 = dems[y1].data[mask.data]
        # keep only pixels valid in both DEMs of the comparison
        both = ~np.ma.getmaskarray(dem1) & ~np.ma.getmaskarray(dem2) & np.isfinite(dem1.filled(np.nan)) & np.isfinite(dem2.filled(np.nan))
        dem1, dem2 = dem1.data[both], dem2.data[both]

        counts1, _ = np.histogram(dem1.flatten(), bins=bins)
        counts2, _ = np.histogram(dem2.flatten(), bins=bins)
        counts1 = (counts1 * np.prod(dems[2019].res)) / 1e6  # Convert to km2
        counts2 = (counts2 * np.prod(dems[2019].res)) / 1e6  # Convert to km2
        band_areas[(f'{y0}-{y1}', name)] = (counts1.copy(), counts2.copy())

rows=[(p,g,f"{bins[k]}-{bins[k+1]}",a[k],b[k],b[k]-a[k]) for (p,g),(a,b) in band_areas.items() for k in range(len(a))]
pd.DataFrame(rows,columns=["period","glacier","band_m","area_start_km2","area_end_km2","d_area_km2"]).to_csv(f"{out_dir}/hypsometry_bands_common.csv",index=False)

## Area per 100 m band, pixels valid in all five DEMs → `hypsometry_bins_5dem_footprint.csv` (Fig. 3)
One common footprint for every DEM, so the band changes of the intervals add up.

In [ ]:
px_km2 = np.prod(dems[2019].res) / 1e6
valid_all = np.logical_and.reduce([np.isfinite(np.ma.filled(dems[y].data.astype(float), np.nan)) for y in list_years])
rows = []
for m, name in zip(masks, mask_names):
    for y in list_years:
        area = np.histogram(dems[y].data[m.data & valid_all].flatten(), bins=bins)[0] * px_km2
        rows += [(name, y, bins[k], area[k]) for k in range(len(area))]
pd.DataFrame(rows, columns=['lobe', 'dem', 'bin_lo', 'area_km2']).to_csv(f'{out_dir}/hypsometry_bins_5dem_footprint.csv', index=False)